In [1]:
import pyterrier as pt
import requests
import pandas as pd
import pickle
import nltk
import numpy as np

In [3]:
query = 'artificial intelligence'
url = f"https://www.bibsonomy.org/json/search/{query}?items=1000"

In [5]:
res = requests.get(url)

In [6]:
data = res.json()

In [7]:
df_ai = pd.DataFrame(data["items"])
df_ai = df_ai[df_ai["type"] == "Publication"]

In [8]:
df_ai_filtered = df_ai.dropna(axis=1, thresh=len(df_ai)*0.5)

In [9]:
df_ai_filtered.columns

Index(['type', 'id', 'tags', 'intraHash', 'label', 'user', 'description',
       'date', 'changeDate', 'count', 'url', 'interHash', 'pub-type', 'year',
       'author', 'authors', 'volume', 'pages', 'bibtexKey', 'publisher'],
      dtype='object')

In [10]:
useful_columns = ['id', 'tags', 'label', 'description', 'date', 'changeDate', 'url', 'pub-type', 'year', 'author', 'authors', 'publisher']
df_ai_filtered = df_ai_filtered[useful_columns]

In [11]:
# initialise pyterrier
if not pt.started():
    pt.init()

/var/folders/88/9blsh45918g8y3n5vl6vztxr0000gq/T/ipykernel_2339/1219906578.py:2: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.10 (build: craigm 2024-08-22 17:33), helper_version=0.0.8]
/var/folders/88/9blsh45918g8y3n5vl6vztxr0000gq/T/ipykernel_2339/1219906578.py:3: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [21]:
df_ai_filtered['docno'] = df_ai_filtered['id']
df_ai_filtered['text'] = df_ai_filtered['label']

In [27]:
index_folder = "./ai_index"

indexer = pt.DFIndexer(index_folder, overwrite=True)
index_ref = indexer.index(df_ai_filtered['text'], df_ai_filtered['docno'])

print(f"path to our index: {index_ref.toString()}") 

/var/folders/88/9blsh45918g8y3n5vl6vztxr0000gq/T/ipykernel_2339/1545029842.py:3: DeprecationWarning: Call to deprecated class DFIndexer. (use pt.terrier.IterDictIndexer().index(dataframe.to_dict(orient='records')) instead) -- Deprecated since version 0.11.0.
  indexer = pt.DFIndexer(index_folder, overwrite=True)


path to our index: ./ai_index/data.properties


In [29]:
index = pt.IndexFactory.of(index_ref)

In [31]:
print(index.getCollectionStatistics().toString())

Number of documents: 1000
Number of terms: 1433
Number of postings: 6208
Number of fields: 0
Number of tokens: 6441
Field names: []
Positions:   false



In [33]:
#term -> term_id Nt Tf
#Nt: In how many documents does the term occur 
#TF: How often does the term occur in total (Termfrequenz)

for kv in index.getLexicon():
    print(f"{kv.getKey()} -> {kv.getValue().toString()}")

0 -> term12 Nt=5 TF=6 maxTF=2 @{0 0 0}
02 -> term569 Nt=1 TF=1 maxTF=1 @{0 8 1}
1 -> term1006 Nt=1 TF=1 maxTF=1 @{0 10 1}
10 -> term1032 Nt=2 TF=2 maxTF=1 @{0 12 5}
10th -> term516 Nt=2 TF=2 maxTF=1 @{0 16 5}
11 -> term1096 Nt=2 TF=2 maxTF=1 @{0 20 7}
11th -> term1166 Nt=4 TF=4 maxTF=1 @{0 25 5}
12 -> term475 Nt=1 TF=1 maxTF=1 @{0 32 7}
12th -> term1091 Nt=4 TF=4 maxTF=1 @{0 34 7}
13 -> term1164 Nt=1 TF=1 maxTF=1 @{0 42 1}
13th -> term1247 Nt=3 TF=3 maxTF=1 @{0 44 5}
14 -> term1031 Nt=1 TF=1 maxTF=1 @{0 50 1}
14th -> term1033 Nt=4 TF=4 maxTF=1 @{0 52 5}
15th -> term1038 Nt=4 TF=4 maxTF=1 @{0 60 3}
16 -> term1314 Nt=1 TF=1 maxTF=1 @{0 67 5}
16th -> term897 Nt=4 TF=4 maxTF=1 @{0 70 1}
17 -> term1163 Nt=1 TF=1 maxTF=1 @{0 77 3}
17th -> term952 Nt=4 TF=4 maxTF=1 @{0 79 7}
18 -> term1255 Nt=2 TF=2 maxTF=1 @{0 86 5}
185 -> term709 Nt=2 TF=2 maxTF=1 @{0 90 5}
18th -> term570 Nt=3 TF=3 maxTF=1 @{0 93 5}
19 -> term209 Nt=4 TF=4 maxTF=1 @{0 99 7}
1955 -> term965 Nt=1 TF=1 maxTF=1 @{0 107 1}
1970

In [39]:
term = "barcelona"
index.getLexicon()[term].toString()

'term1123 Nt=3 TF=3 maxTF=1 @{0 1391 1}'

In [41]:
#how often does the term occur
term_freq_dict = {}

for kv in index.getLexicon():
    term_freq_dict[kv.getKey()] = kv.getValue().frequency

In [43]:
#sort all terms from the index in descending order
term_freq_dict_sorted = sorted(term_freq_dict.items(), key=lambda x: x[1], reverse=True)

In [45]:
#show the 30 most popular entries
top_k = 30
term_freq_dict_sorted[:top_k]

[('intellig', 963),
 ('artifici', 931),
 ('confer', 75),
 ('proceed', 66),
 ('ai', 63),
 ('system', 63),
 ('base', 54),
 ('learn', 50),
 ('applic', 49),
 ('approach', 45),
 ('us', 43),
 ('human', 40),
 ('gener', 38),
 ('research', 36),
 ('comput', 35),
 ('educ', 30),
 ('develop', 29),
 ('nation', 28),
 ('analysi', 27),
 ('technolog', 27),
 ('review', 26),
 ('advanc', 24),
 ('impact', 24),
 ('model', 24),
 ('intern', 23),
 ('machin', 23),
 ('modern', 23),
 ('futur', 22),
 ('data', 20),
 ('explain', 20)]

In [47]:
#define our search engine
search_engine = pt.BatchRetrieve(index, wmodel="Tf")

/var/folders/88/9blsh45918g8y3n5vl6vztxr0000gq/T/ipykernel_2339/2780833963.py:2: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  search_engine = pt.BatchRetrieve(index, wmodel="Tf")


In [49]:
search_engine.search("system")

,qid,docid,docno,rank,score,query
0,1,290,https://www.bibsonomy.org/bibtex/24768ef52a6c0...,0,2.0,system
1,1,41,https://www.bibsonomy.org/bibtex/249f58604f738...,1,1.0,system
2,1,68,https://www.bibsonomy.org/bibtex/2f693b1d1c623...,2,1.0,system
3,1,81,https://www.bibsonomy.org/bibtex/28585af0407b9...,3,1.0,system
4,1,84,https://www.bibsonomy.org/bibtex/2f6f3aecbaa29...,4,1.0,system
...,...,...,...,...,...,...
57,1,942,https://www.bibsonomy.org/bibtex/293fb4fd88de7...,57,1.0,system
58,1,951,https://www.bibsonomy.org/bibtex/20dac3cd23401...,58,1.0,system
59,1,974,https://www.bibsonomy.org/bibtex/27ba4173ca906...,59,1.0,system
60,1,980,https://www.bibsonomy.org/bibtex/234996efa89f2...,60,1.0,system


In [51]:
search_engine.search("intelligent education")

,qid,docid,docno,rank,score,query
0,1,736,https://www.bibsonomy.org/bibtex/2b476e9f37ca1...,0,5.0,intelligent education
1,1,241,https://www.bibsonomy.org/bibtex/2f91c0697115d...,1,3.0,intelligent education
2,1,347,https://www.bibsonomy.org/bibtex/26895e4605d5e...,2,3.0,intelligent education
3,1,559,https://www.bibsonomy.org/bibtex/2362d4d486a4a...,3,3.0,intelligent education
4,1,644,https://www.bibsonomy.org/bibtex/226a944b1f244...,4,3.0,intelligent education
...,...,...,...,...,...,...
870,1,994,https://www.bibsonomy.org/bibtex/2d968ca695d80...,870,1.0,intelligent education
871,1,995,https://www.bibsonomy.org/bibtex/2506d96485fc4...,871,1.0,intelligent education
872,1,996,https://www.bibsonomy.org/bibtex/28765d3d19da8...,872,1.0,intelligent education
873,1,997,https://www.bibsonomy.org/bibtex/22405228e10c2...,873,1.0,intelligent education


In [55]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

In [57]:
stem = stemmer.stem("learning")
stem

'learn'

In [59]:
lexicon = index.getLexicon()
lexicon[stem].getDocumentFrequency()

48

In [61]:
#how many documents occur in our index?
index.getCollectionStatistics().numberOfDocuments

1000

In [73]:
def get_idf_for_term(term, index):
  lex = index.getLexicon()
  stemmed_term = stemmer.stem(term)

  if not stemmed_term in lex:
    return 
    
  lex_entry = lex[stemmed_term]

  df_term = lex_entry.getDocumentFrequency()
  N = index.getCollectionStatistics().numberOfDocuments
  
  #inverse document frequency
  idf = N/df_term

  #apply logarithm(base 10) to idf
  log_idf = np.emath.logn(10, idf)

  return log_idf

In [75]:
get_idf_for_term("learning", index)

1.3187587626244126

In [77]:
#define your own tf_idf method here:

def calc_tf_idf(query, docno, index):
    #remember that tfidf is the product of two components
    #hint: the tf model search result contains tf frequencies
    return 0

In [87]:
search_engine_tfidf = pt.BatchRetrieve(index, wmodel="TF_IDF")


/var/folders/88/9blsh45918g8y3n5vl6vztxr0000gq/T/ipykernel_2339/1376347478.py:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  search_engine_tfidf = pt.BatchRetrieve(index, wmodel="TF_IDF")


In [89]:
search_engine_tfidf.search("intelligent education")

,qid,docid,docno,rank,score,query
0,1,815,https://www.bibsonomy.org/bibtex/29cc0434b3e50...,0,4.361531,intelligent education
1,1,926,https://www.bibsonomy.org/bibtex/275d5c245ba41...,1,4.361531,intelligent education
2,1,972,https://www.bibsonomy.org/bibtex/2eb8e8e572869...,2,4.361531,intelligent education
3,1,24,https://www.bibsonomy.org/bibtex/259133e526641...,3,4.033687,intelligent education
4,1,78,https://www.bibsonomy.org/bibtex/259133e526641...,4,4.033687,intelligent education
...,...,...,...,...,...,...
870,1,132,https://www.bibsonomy.org/bibtex/2c4df47ce17b7...,870,0.373180,intelligent education
871,1,230,https://www.bibsonomy.org/bibtex/29fcc0f646278...,871,0.373180,intelligent education
872,1,595,https://www.bibsonomy.org/bibtex/21b69569af8d0...,872,0.373180,intelligent education
873,1,250,https://www.bibsonomy.org/bibtex/2ba15da18da2c...,873,0.345844,intelligent education


In [91]:
df_ai_filtered.columns

Index(['id', 'tags', 'label', 'description', 'date', 'changeDate', 'url',
       'pub-type', 'year', 'author', 'authors', 'publisher', 'docno', 'text'],
      dtype='object')

In [93]:
ai_dict = df_ai_filtered.to_dict(orient='records')

In [95]:
ai_dict

[{'id': 'https://www.bibsonomy.org/bibtex/24e338b04bc2abafd9fa0a1f4fe79103a/ijtsrd',
  'tags': ['710-714'],
  'label': 'Artificial Intelligence in Agriculture',
  'description': '',
  'date': '2021-04-13 13:15:04',
  'changeDate': '2021-04-13 13:15:04',
  'url': 'https://www.ijtsrd.com/engineering/electrical-engineering/38513/artificial-intelligence-in-agriculture/matthew-n-o-sadiku',
  'pub-type': 'article',
  'year': '2021',
  'author': ['Matthew N. O. Sadiku | Sarhan M. Musa | Abayomi Ajayi-Majebi'],
  'authors': [{'first': 'Matthew N. O. Sadiku | Sarhan M. Musa | Abayomi',
    'last': 'Ajayi-Majebi'}],
  'publisher': nan,
  'docno': 'https://www.bibsonomy.org/bibtex/24e338b04bc2abafd9fa0a1f4fe79103a/ijtsrd',
  'text': 'Artificial Intelligence in Agriculture'},
 {'id': 'https://www.bibsonomy.org/bibtex/283c4174f111d968e8ca51b252f8fea3e/meneteqel',
  'tags': ['digitalisation',
   'artificial_intelligence',
   'society',
   'labour_rights'],
  'label': 'Artificial intelligence, labour

In [99]:
index_folder_mult = "./ai_index_mult"

fields=['docno', 'text', 'tags', 'description']

indexer_mult = pt.IterDictIndexer(index_folder_mult, meta={'docno': 200, 'text': 4096}, overwrite=True)
index_ref_mult = indexer_mult.index(ai_dict, fields=fields)

print(f"path to our index: {index_ref_mult.toString()}")

path to our index: ./ai_index_mult/data.properties


In [101]:
index_mult = pt.IndexFactory.of(index_ref_mult)

In [103]:
print(index_mult.getCollectionStatistics().toString())

Number of documents: 1000
Number of terms: 2273
Number of postings: 14595
Number of fields: 4
Number of tokens: 16828
Field names: [docno, text, tags, description]
Positions:   false



In [107]:
search_engine_mult = pt.BatchRetrieve(index_mult, wmodel="TF_IDF")

/var/folders/88/9blsh45918g8y3n5vl6vztxr0000gq/T/ipykernel_2339/495383578.py:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  search_engine_mult = pt.BatchRetrieve(index_mult, wmodel="TF_IDF")


In [109]:
res = search_engine_mult.search("intelligence")
res

,qid,docid,docno,rank,score,query
0,1,732,https://www.bibsonomy.org/bibtex/224ac67b53792...,0,0.949091,intelligence
1,1,347,https://www.bibsonomy.org/bibtex/26895e4605d5e...,1,0.924343,intelligence
2,1,323,https://www.bibsonomy.org/bibtex/29a73b5ea0550...,2,0.891737,intelligence
3,1,501,https://www.bibsonomy.org/bibtex/2f7cf941158e2...,3,0.891737,intelligence
4,1,733,https://www.bibsonomy.org/bibtex/21a922cb4af1f...,4,0.891737,intelligence
...,...,...,...,...,...,...
925,1,860,https://www.bibsonomy.org/bibtex/25a061b694a47...,925,0.385489,intelligence
926,1,833,https://www.bibsonomy.org/bibtex/25a061b694a47...,926,0.379302,intelligence
927,1,666,https://www.bibsonomy.org/bibtex/28cff9d35fb12...,927,0.367505,intelligence
928,1,667,https://www.bibsonomy.org/bibtex/28cff9d35fb12...,928,0.367505,intelligence


In [111]:
#save our data
pickle.dump(df_ai_filtered, open("workspace/ai_publications.pkl", "wb"))

FileNotFoundError: [Errno 2] No such file or directory: 'workspace/ai_publications.pkl'

In [113]:
pip install streamlit

Note: you may need to restart the kernel to use updated packages.
